# Unified ORF coding evaluation and Top-K workflow

一次运行完成共享预处理、cell-aware ORF 匹配、综合指标、特征组合评估和 Top-K 数据生成。Top-K 绘图保持独立，并且只输出 PDF。

In [ ]:
from pathlib import Path
import sys

trace_root = Path("/home/user/data3/rbase/translation_model/TRACE")
sys.path.insert(0, str(trace_root / "src"))

from eval.orf_coding_performance import (
    calculate_top_k_precision,
    evaluate_orf_level_predictions,
    plot_top_k_precision,
    plot_top_k_recall,
)
from model.translation_predictor import get_active_transcripts

## 单个细胞环境：一次完成全面评估和 Top-K 计算

In [ ]:
project_root = Path("/home/user/data3/rbase/translation_model")
out_dir = project_root / "results/in-house/unified_run"
pred_file = project_root / "results/in-house/high_confidence_orfs.brain_fetal_inhouse.balanced_mode.csv"
ms_gt_file = project_root / "data/in-house/ms_orf_gt.brain_fetal_inhouse.csv"

# Replace this with the transcript array used during profile prediction.
isoform_expressed = [...]

top_k_score_definition = {
    "Base_Score": "base_expr_score",
    "Features": "uniformity_of_signal+step_up_contrast",
    "Method": "product",
}

results = evaluate_orf_level_predictions(
    pred_csv_paths=[str(pred_file)],
    gt_csv_paths={"brain_fetal_inhouse": str(ms_gt_file)},
    target_transcript_ids=isoform_expressed,
    min_orf_len=21,
    max_orf_len=450,
    out_dir=str(out_dir),
    overlap_threshold=0.95,
    callable_start_codons=["ATG"],
    target_score_col="expr_score",
    evaluate_score_combinations=True,
    combination_top_k_values=[100, 500, 1000, 2000, 5000],
    combination_primary_metric="Precision_at_1000",
    top_k_combined_score=top_k_score_definition,
)

results["top_k_summary"]

## 独立绘制 Top-K PDF

In [ ]:
top_k_df = results["top_k"]

plot_top_k_precision(top_k_df, out_dir=str(out_dir), max_k=5000)
plot_top_k_recall(top_k_df, out_dir=str(out_dir), max_k=5000)
plot_top_k_precision(
    top_k_df,
    out_dir=str(out_dir),
    max_k=5000,
    rank_scope="cell_type",
)

## 稍后从 unified table 重新排序，不重新匹配 ORF

In [ ]:
reloaded_top_k = calculate_top_k_precision(
    evaluation_csv_path=str(out_dir / "unified_evaluation_table.csv"),
    combined_score=top_k_score_definition,
)

plot_top_k_precision(
    reloaded_top_k,
    out_dir=str(out_dir),
    max_k=5000,
)

## 多细胞类型：使用 cell-type-specific callable transcript 字典

In [ ]:
cell_types = ["brain_cerebrum", "liver", "prostate", "kidney", "testis"]
active_transcripts = get_active_transcripts(
    tpm_csv_path=str(project_root / "models/lib/human_expression_tpm.csv"),
    mapping_csv_path="/home/user/data3/rbase/genome_ref/Homo_sapiens/hg38/ens_genes_v112.txt",
    cell_type=cell_types,
    min_tpm=1,
)

public_results = evaluate_orf_level_predictions(
    pred_csv_paths=[
        str(project_root / "results/coding_task/public_ms/brain_cerebrum/high_confidence_orfs.brain_cerebrum.long_mode.csv"),
        str(project_root / "results/coding_task/public_ms/liver/high_confidence_orfs.liver.long_mode.csv"),
    ],
    gt_csv_paths={
        "brain_cerebrum": str(project_root / "data/public_ms/ms_orf_gt.brain.csv"),
        "liver": str(project_root / "data/public_ms/ms_orf_gt.liver.csv"),
    },
    target_transcript_ids=active_transcripts,
    min_orf_len=21,
    max_orf_len=450,
    out_dir=str(project_root / "results/coding_task/public_ms/unified_run"),
    overlap_threshold=0.95,
    callable_start_codons=["ATG"],
    target_score_col="expr_score",
    evaluate_score_combinations=True,
    top_k_combined_score=top_k_score_definition,
)

public_results["top_k_summary"]

## 与其他 ORF caller 比较 Recall@K

In [ ]:
from plot.benchmark_orf_ident import plot_multi_model_top_k_recall

benchmark_manifest = [
    {
        "model": "TRACE",
        "path": str(out_dir / "unified_evaluation_table.csv"),
        "score_col": "__top_k_combined_score",
    },
    {
        "model": "RiboTIE",
        "path": str(project_root / "results/benchmark/coding/RiboTIE/unified_evaluation_table.csv"),
        "score_col": "score",
    },
    {
        "model": "RiboTISH",
        "path": str(project_root / "results/benchmark/coding/RiboTISH/unified_evaluation_table.csv"),
        "score_col": "score",
    },
]

recall_curve_df, recall_pdf = plot_multi_model_top_k_recall(
    manifest=benchmark_manifest,
    out_dir=str(project_root / "results/benchmark/coding"),
    min_k=10,
    max_k=5000,
    suffix="brain_fetal",
    require_same_total_gt=True,
)

recall_pdf